In [1]:
import pandas as pd
import polars as pl
import os

In [2]:
DATASET_PATH = '../dataset/E003/'

# Loading the Dataset

## Load the BioMart Dataset

In [3]:
col_names = ['ensembl_gene_id',
                'external_gene_name',
                'chromosome_name',
                'start_position',
                'end_position',
                'strand']

biomart_df = pd.read_csv("biomart_response.txt", sep='\t', names = col_names)

In [4]:
print(biomart_df.head())
print(biomart_df.shape)

   ensembl_gene_id external_gene_name chromosome_name  start_position  \
0  ENSG00000000003             TSPAN6               X        99883667   
1  ENSG00000000005               TNMD               X        99839799   
2  ENSG00000000419               DPM1              20        49551404   
3  ENSG00000000457              SCYL3               1       169818772   
4  ENSG00000000460           C1orf112               1       169631245   

   end_position  strand  
0      99894988      -1  
1      99854882       1  
2      49575092      -1  
3     169863408      -1  
4     169823221       1  
(19645, 6)


In [5]:
# Calculate the TSS

biomart_df['tss'] = biomart_df.apply(lambda row: row['start_position'] if row['strand'] == 1 else row['end_position'], axis=1)

In [6]:
# Reformat the chromosome name
biomart_df['chromosome_name'] = 'chr' + biomart_df['chromosome_name'].astype(str)

In [7]:
biomart_df.head()

,ensembl_gene_id,external_gene_name,chromosome_name,start_position,end_position,strand,tss
0,ENSG00000000003,TSPAN6,chrX,99883667,99894988,-1,99894988
1,ENSG00000000005,TNMD,chrX,99839799,99854882,1,99839799
2,ENSG00000000419,DPM1,chr20,49551404,49575092,-1,49575092
3,ENSG00000000457,SCYL3,chr1,169818772,169863408,-1,169863408
4,ENSG00000000460,C1orf112,chr1,169631245,169823221,1,169631245


In [8]:
biomart_df = biomart_df.rename(columns={'ensembl_gene_id': 'gene_id'})

In [9]:
biomart_df.head()

,gene_id,external_gene_name,chromosome_name,start_position,end_position,strand,tss
0,ENSG00000000003,TSPAN6,chrX,99883667,99894988,-1,99894988
1,ENSG00000000005,TNMD,chrX,99839799,99854882,1,99839799
2,ENSG00000000419,DPM1,chr20,49551404,49575092,-1,49575092
3,ENSG00000000457,SCYL3,chr1,169818772,169863408,-1,169863408
4,ENSG00000000460,C1orf112,chr1,169631245,169823221,1,169631245


In [10]:
biomart_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19645 entries, 0 to 19644
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   gene_id             19645 non-null  object
 1   external_gene_name  19645 non-null  object
 2   chromosome_name     19645 non-null  object
 3   start_position      19645 non-null  int64 
 4   end_position        19645 non-null  int64 
 5   strand              19645 non-null  int64 
 6   tss                 19645 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 1.0+ MB


In [11]:
biomart_df.to_csv('ensg.csv', header=True, index=False)

## Loading the gene expression dataset

In [12]:
gene_exp = pl.read_csv(os.path.join(DATASET_PATH, "57epigenomes.RPKM.pc"), separator='\t', truncate_ragged_lines=True)

In [13]:
gene_exp.head()

gene_id,E000,E003,E004,E005,E006,E007,E011,E012,E013,E016,E024,E027,E028,E037,E038,E047,E050,E053,E054,E055,E056,E057,E058,E059,E061,E062,E065,E066,E070,E071,E079,E082,E084,E085,E087,E094,E095,E096,E097,E098,E100,E104,E105,E106,E109,E112,E113,E114,E116,E117,E118,E119,E120,E122,E123,E127,E128
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000000003""",23.265,43.985,37.413,29.459,21.864,55.649,52.94,71.629,61.292,44.28,63.184,7.49,8.541,0.576,1.393,1.235,5.544,15.933,27.15,6.433,3.812,6.257,10.151,8.898,14.658,0.298,5.605,73.205,20.954,7.645,35.083,6.265,53.039,64.971,9.594,14.46,3.122,13.463,54.677,13.735,2.435,8.833,4.494,36.012,19.252,11.928,5.637,37.989,0.038,42.639,49.983,11.554,11.847,43.723,0.267,13.758,15.818
"""ENSG00000000005""",0.872,1.642,6.498,0.0,0.157,0.003,0.115,0.087,0.055,1.577,0.726,0.0,0.0,0.0,0.0,0.029,0.0,0.051,0.07,0.0,0.0,0.0,0.0,0.0,0.006,0.0,0.0,0.191,0.0,0.018,0.251,0.0,0.566,0.336,0.03,0.0,0.07,0.0,10.67,0.424,0.032,0.524,0.092,0.205,0.134,0.678,0.121,0.0,0.0,0.0,0.0,0.0,0.018,0.0,0.006,0.0,0.0
"""ENSG00000000419""",55.208,35.259,58.308,48.208,37.477,45.923,44.959,40.438,41.97,51.515,35.129,63.304,47.743,45.394,47.041,38.384,37.351,11.078,12.225,27.126,21.572,28.648,44.444,13.179,22.421,28.648,52.753,52.609,15.701,21.769,26.467,7.879,29.927,30.095,32.469,56.167,32.202,26.051,42.731,19.683,67.684,46.172,33.687,39.226,47.562,61.359,54.866,52.215,79.197,107.098,62.811,42.386,54.869,16.652,73.719,56.578,56.371
"""ENSG00000000457""",3.237,2.596,2.345,8.775,2.723,3.7,3.912,5.011,4.158,3.292,3.16,3.683,2.532,7.409,8.577,7.853,17.602,3.295,4.301,3.706,1.523,1.602,3.933,1.877,4.641,5.433,3.417,4.733,3.349,2.222,5.325,2.977,8.497,9.679,5.593,5.731,2.622,3.907,7.649,5.455,10.873,2.529,2.811,6.044,4.526,8.791,5.484,4.829,11.082,8.814,2.646,2.483,2.527,2.549,7.651,4.967,3.714
"""ENSG00000000460""",7.299,6.649,7.838,7.324,0.83,5.354,5.94,5.704,6.213,7.551,7.705,1.04,1.213,3.459,3.813,3.424,6.91,3.855,4.401,2.896,3.246,3.31,6.491,2.782,2.799,2.292,1.137,0.942,4.716,1.12,1.487,1.611,3.408,3.58,0.806,1.25,0.47,1.134,1.69,1.126,0.518,0.717,0.694,1.893,1.952,3.137,1.631,8.001,13.743,25.369,3.373,4.646,2.179,4.099,22.103,3.29,2.491


In [14]:
gene_exp_df = gene_exp.to_pandas()

In [15]:
gene_exp_df.head()

,gene_id,E000,E003,E004,E005,E006,E007,E011,E012,E013,...,E114,E116,E117,E118,E119,E120,E122,E123,E127,E128
0,ENSG00000000003,23.265,43.985,37.413,29.459,21.864,55.649,52.940,71.629,61.292,...,37.989,0.038,42.639,49.983,11.554,11.847,43.723,0.267,13.758,15.818
1,ENSG00000000005,0.872,1.642,6.498,0.000,0.157,0.003,0.115,0.087,0.055,...,0.000,0.000,0.000,0.000,0.000,0.018,0.000,0.006,0.000,0.000
2,ENSG00000000419,55.208,35.259,58.308,48.208,37.477,45.923,44.959,40.438,41.970,...,52.215,79.197,107.098,62.811,42.386,54.869,16.652,73.719,56.578,56.371
3,ENSG00000000457,3.237,2.596,2.345,8.775,2.723,3.700,3.912,5.011,4.158,...,4.829,11.082,8.814,2.646,2.483,2.527,2.549,7.651,4.967,3.714
4,ENSG00000000460,7.299,6.649,7.838,7.324,0.830,5.354,5.940,5.704,6.213,...,8.001,13.743,25.369,3.373,4.646,2.179,4.099,22.103,3.290,2.491


In [16]:
gene_exp_df.shape

(19795, 58)

## Select only valid genes

In [17]:
# Select only valid genes
gene_exp_df = gene_exp_df[gene_exp_df['gene_id'].isin(biomart_df['gene_id'])]

In [18]:
gene_exp_df.head()

,gene_id,E000,E003,E004,E005,E006,E007,E011,E012,E013,...,E114,E116,E117,E118,E119,E120,E122,E123,E127,E128
0,ENSG00000000003,23.265,43.985,37.413,29.459,21.864,55.649,52.940,71.629,61.292,...,37.989,0.038,42.639,49.983,11.554,11.847,43.723,0.267,13.758,15.818
1,ENSG00000000005,0.872,1.642,6.498,0.000,0.157,0.003,0.115,0.087,0.055,...,0.000,0.000,0.000,0.000,0.000,0.018,0.000,0.006,0.000,0.000
2,ENSG00000000419,55.208,35.259,58.308,48.208,37.477,45.923,44.959,40.438,41.970,...,52.215,79.197,107.098,62.811,42.386,54.869,16.652,73.719,56.578,56.371
3,ENSG00000000457,3.237,2.596,2.345,8.775,2.723,3.700,3.912,5.011,4.158,...,4.829,11.082,8.814,2.646,2.483,2.527,2.549,7.651,4.967,3.714
4,ENSG00000000460,7.299,6.649,7.838,7.324,0.830,5.354,5.940,5.704,6.213,...,8.001,13.743,25.369,3.373,4.646,2.179,4.099,22.103,3.290,2.491


In [19]:
gene_exp_df.shape

(19645, 58)

In [22]:
# Saving the gene expression
gene_exp_df.to_csv("57epigenomes_clean.csv", header=True, index=False)

# Gene Selection E066 (liver cell)

In [71]:
# Select the E066 (liver cell)
E066_df = gene_exp_df[["gene_id", "E066"]]

In [72]:
E066_df.head()

,gene_id,E066
0,ENSG00000000003,73.205
1,ENSG00000000005,0.191
2,ENSG00000000419,52.609
3,ENSG00000000457,4.733
4,ENSG00000000460,0.942


## Adding the label

In [73]:
# Find the median
E066_median = E066_df["E066"].median()
print(E066_median)

2.423


In [74]:
# Create a label column
E066_df.loc[:, 'label'] = E066_df['E066'].apply(lambda x: 1 if x >= E066_median else -1)

/tmp/ipykernel_1470113/3870573093.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  E066_df.loc[:, 'label'] = E066_df['E066'].apply(lambda x: 1 if x >= E066_median else -1)


In [75]:
E066_df.head()

,gene_id,E066,label
0,ENSG00000000003,73.205,1
1,ENSG00000000005,0.191,-1
2,ENSG00000000419,52.609,1
3,ENSG00000000457,4.733,1
4,ENSG00000000460,0.942,-1


In [76]:
E066_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19645 entries, 0 to 19792
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   gene_id  19645 non-null  object 
 1   E066     19645 non-null  float64
 2   label    19645 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB


In [77]:
E066_df.shape

(19645, 3)

## Join with ENSG data

In [78]:
# Join with ENSG data
E066_merged_df = E066_df.merge(biomart_df, on='gene_id')

In [79]:
E066_merged_df.head()

,gene_id,E066,label,external_gene_name,chromosome_name,start_position,end_position,strand,tss
0,ENSG00000000003,73.205,1,TSPAN6,chrX,99883667,99894988,-1,99894988
1,ENSG00000000005,0.191,-1,TNMD,chrX,99839799,99854882,1,99839799
2,ENSG00000000419,52.609,1,DPM1,chr20,49551404,49575092,-1,49575092
3,ENSG00000000457,4.733,1,SCYL3,chr1,169818772,169863408,-1,169863408
4,ENSG00000000460,0.942,-1,C1orf112,chr1,169631245,169823221,1,169631245


In [80]:
E066_merged_df.to_csv("E066_merged.csv", header=True, index=False)

# Preparing the data for bedtools intersect

In [117]:
# Find the +/- 5,000 bp location from TSS
# Make sure that the start is smaller than the end

E066_merged_df['start'] = E066_merged_df['tss'] - 5000
E066_merged_df['end'] = E066_merged_df['tss'] + 5000

In [118]:
E066_merged_df.head()

,chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
0,chrX,99889988,99899988,ENSG00000000003,73.205,-1,1,TSPAN6,99883667,99894988,99894988
1,chrX,99834799,99844799,ENSG00000000005,0.191,1,-1,TNMD,99839799,99854882,99839799
2,chr20,49570092,49580092,ENSG00000000419,52.609,-1,1,DPM1,49551404,49575092,49575092
3,chr1,169858408,169868408,ENSG00000000457,4.733,-1,1,SCYL3,169818772,169863408,169863408
4,chr1,169626245,169636245,ENSG00000000460,0.942,1,-1,C1orf112,169631245,169823221,169631245


In [119]:
# Reorder the column
E066_merged_df = E066_merged_df[['chromosome_name',
                                'start',
                                'end',
                                'gene_id',
                                'E066',
                                'strand',
                                'label',
                                'external_gene_name',
                                'start_position',
                                'end_position',
                                'tss' 
                                ]]

In [120]:
E066_merged_df.head()

,chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
0,chrX,99889988,99899988,ENSG00000000003,73.205,-1,1,TSPAN6,99883667,99894988,99894988
1,chrX,99834799,99844799,ENSG00000000005,0.191,1,-1,TNMD,99839799,99854882,99839799
2,chr20,49570092,49580092,ENSG00000000419,52.609,-1,1,DPM1,49551404,49575092,49575092
3,chr1,169858408,169868408,ENSG00000000457,4.733,-1,1,SCYL3,169818772,169863408,169863408
4,chr1,169626245,169636245,ENSG00000000460,0.942,1,-1,C1orf112,169631245,169823221,169631245


In [122]:
# Save to CSV
E066_merged_df.to_csv('E066.bed', header=False, index=False, sep='\t')

In [123]:
# Sample for bedtools
E066_merged_df.head(10).to_csv('E066_10.bed', header=False, index=False, sep='\t')